# 01 — Carregar e extrair o GSM8K

Objetivo: baixar os splits `main` do GSM8K e extrair enunciado, passos, contas anotadas e resposta final. Entrada: dataset Hugging Face. Saída: `data/raw/gsm8k_extraido.jsonl`. O corpus é mantido completo nesta etapa; `max_examples` limita a amostra nos notebooks seguintes.

In [13]:
# Opcional: %pip install datasets pydantic>=2.7
import json
import random
import re
from pathlib import Path
from typing import Any

from datasets import load_dataset
from pydantic import BaseModel, ConfigDict

def localizar_projeto() -> Path:
    '''Localiza a pasta do pipeline mesmo quando o kernel inicia na raiz do repo.'''
    candidatos = [Path.cwd(), *Path.cwd().parents]
    for base in candidatos:
        pasta = base / 'notebooks' / 'eja_diagnosticador_dataset'
        if (pasta / 'data' / 'config.json').exists():
            return pasta
        if base.name == 'eja_diagnosticador_dataset' and (base / 'data' / 'config.json').exists():
            return base
    raise FileNotFoundError('Não encontrei notebooks/eja_diagnosticador_dataset/data/config.json')

ROOT = localizar_projeto()
CONFIG = json.loads((ROOT / 'data' / 'config.json').read_text(encoding='utf-8'))
random.seed(CONFIG['seed'])
SAIDA = ROOT / 'data' / 'raw' / 'gsm8k_extraido.jsonl'
PADRAO_CONTA = re.compile(r'<<(.+?)=(.+?)>>')

As anotações do GSM8K são preservadas como texto, sem executar expressões nesta etapa. Soluções sem marcador final ou enunciado são contabilizadas como rejeitadas, em vez de receberem um valor presumido.

In [14]:
class RegistroExtraido(BaseModel):
    '''Esquema validado para cada linha derivada do corpus.'''
    model_config = ConfigDict(extra='forbid')
    id: str
    base_id: str
    split_origem: str
    pergunta_original: str
    solucao_original: str
    passos_raciocinio: list[str]
    contas_anotadas: list[dict[str, str]]
    resposta_final: str

def serializar_jsonl(registro: dict[str, Any]) -> str:
    '''Escapa separadores Unicode que str.splitlines também trata como quebra.'''
    texto = json.dumps(registro, ensure_ascii=False)
    for caractere, escape in ((chr(0x85), r'\u0085'), (chr(0x2028), r'\u2028'), (chr(0x2029), r'\u2029')):
        texto = texto.replace(caractere, escape)
    return texto

def extrair_registro(split: str, indice: int, linha: dict[str, Any]) -> dict[str, Any] | None:
    '''Extrai um exemplo GSM8K sem interpretar ou executar a solução.'''
    pergunta = str(linha.get('question', '')).strip()
    solucao = str(linha.get('answer', '')).strip()
    partes = re.split(r'####\s*', solucao, maxsplit=1)
    if not pergunta or len(partes) != 2 or not partes[1].strip():
        return None
    corpo, resposta = partes[0].strip(), partes[1].strip()
    contas = [{'expressao': m.group(1).strip(), 'resultado': m.group(2).strip()}
              for m in PADRAO_CONTA.finditer(corpo)]
    passos = []
    for linha_passo in corpo.splitlines():
        passo = re.sub(r'<<(.+?)=(.+?)>>', '', linha_passo)
        passo = ' '.join(passo.split())
        if passo:
            passos.append(passo)
    registro = {'id': f'{split}:{indice}', 'base_id': f'{split}:{indice}',
            'split_origem': split, 'pergunta_original': pergunta,
            'solucao_original': solucao, 'passos_raciocinio': passos,
            'contas_anotadas': contas, 'resposta_final': resposta}
    return RegistroExtraido.model_validate(registro).model_dump()

dataset = load_dataset('openai/gsm8k', 'main')
SAIDA.parent.mkdir(parents=True, exist_ok=True)
contagens = {}
descartados = 0
with SAIDA.open('w', encoding='utf-8') as arquivo:
    for split in ('train', 'test'):
        total = 0
        for indice, linha in enumerate(dataset[split]):
            registro = extrair_registro(split, indice, linha)
            if registro is None:
                descartados += 1
                continue
            arquivo.write(serializar_jsonl(registro) + '\n')
            total += 1
        contagens[split] = total
print(f'Arquivo: {SAIDA}; registros: {contagens}; descartados: {descartados}')

Arquivo: c:\Users\maximus.nascimento\Downloads\calma-ai\notebooks\eja_diagnosticador_dataset\data\raw\gsm8k_extraido.jsonl; registros: {'train': 7473, 'test': 1319}; descartados: 0


## Verificação
Confirme a cobertura dos splits, a presença de respostas e contas anotadas, e inspecione um registro bruto. O arquivo deve existir antes de avançar.

In [15]:
registros = [json.loads(linha) for linha in SAIDA.read_text(encoding='utf-8').splitlines() if linha]
assert registros and all(r['pergunta_original'] and r['resposta_final'] for r in registros)
print('Total:', len(registros), '| por split:', {s: sum(r['split_origem'] == s for r in registros) for s in ('train', 'test')})
print('Com contas anotadas:', sum(bool(r['contas_anotadas']) for r in registros), '| descartados:', descartados)
print(json.dumps(registros[0], ensure_ascii=False, indent=2)[:2500])

Total: 8792 | por split: {'train': 7473, 'test': 1319}
Com contas anotadas: 8679 | descartados: 0
{
  "id": "train:0",
  "base_id": "train:0",
  "split_origem": "train",
  "pergunta_original": "Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?",
  "solucao_original": "Natalia sold 48/2 = <<48/2=24>>24 clips in May.\nNatalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.\n#### 72",
  "passos_raciocinio": [
    "Natalia sold 48/2 = 24 clips in May.",
    "Natalia sold 48+24 = 72 clips altogether in April and May."
  ],
  "contas_anotadas": [
    {
      "expressao": "48/2",
      "resultado": "24"
    },
    {
      "expressao": "48+24",
      "resultado": "72"
    }
  ],
  "resposta_final": "72"
}
